In [0]:
employee_df = spark.read.format("delta").load("/Volumes/workspace/default/employee_raw_vol/silver/Employee_info/dim_employee")

display(employee_df)

In [0]:
salary_df = employee_df.groupBy("department_id") \
.sum("salary") \
.withColumnRenamed("sum(salary)", "total_salary") \
.orderBy("total_salary", ascending=False)

In [0]:
emp_count_df = employee_df.groupBy("department_id","country_id") \
.count() \
.withColumnRenamed("count","employee_count")

In [0]:
department_df = spark.read.option("header","true") \
.csv("/Volumes/workspace/default/employee_raw_vol/source_to_bronze/department")

country_df = spark.read.option("header","true") \
.csv("/Volumes/workspace/default/employee_raw_vol/source_to_bronze/country")

In [0]:
dept_country_df = department_df.join(country_df,"CountryID") \
.select("DepartmentName","CountryName")

In [0]:
avg_age_df = employee_df.groupBy("department_id") \
.avg("age") \
.withColumnRenamed("avg(age)","avg_age")

In [0]:
from pyspark.sql.functions import current_date

salary_df = salary_df.withColumn("at_load_date",current_date())

In [0]:
gold_path = "/Volumes/workspace/default/employee_raw_vol/gold/employee/fact_employee"

salary_df.write \
.format("delta") \
.mode("overwrite") \
.save(gold_path)

In [0]:
gold_df = spark.read.format("delta").load(
"/Volumes/workspace/default/employee_raw_vol/gold/employee/fact_employee"
)

display(gold_df)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/employee_raw_vol/gold/employee"))